# Chapter 1 · Why agents fail: the workbench

A coding agent in the repository of the North Cycles store: thirteen Python modules, 59 tests and fourteen tasks written as someone at the store would write them, each with hidden tests the agent does not see. First, the agent with only the request. Then, with a minimal workbench: an `AGENTS.md` with the definition of done and a gate that checks it before accepting a "done".

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

The model takes about 8 GB; the first time it takes a few minutes to download. `prepare` downloads from the book's repository the store's program, the tasks and the tools module.

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 1: why agents fail, and the minimal workbench."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The tools, without a model

A copy of the repository that is deleted when leaving the `with`, and the tools the agent will use. Notice the error from `edit` when the piece does not match, and how it rejects an edit that would break the file.

In [ ]:
with rt.Workspace() as workspace:
    t = rt.Tools(workspace)
    print(t.call("search", {"pattern": r"def zone",
                            "folder": "shipping.py"}))
    print(t.call("read_file", {"path": "shipping.py",
                               "start": 6, "end": 15}))
    print(t.call("edit", {"path": "shipping.py",
                          "old": "code = str(zip_code)",
                          "new": "  code = str(zip_code)"}))
    print(t.call("run_tests")["result"])

A task from the inside: the request, the scope contract and its hidden tests.

In [ ]:
task = rt.read_task("T02")
print(task["request"])
print(task["contract"])
hidden = task["folder"] / "hidden_tests"
print(sorted(p.name for p in hidden.iterdir()))

## 3. The model

The template with the tools, generation, reading the calls and trimming the conversation when it does not fit on the GPU.

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with greedy decoding. Returns the text and the
    input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_TOKENS_STEP,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, prefill_chunk_size=1024,
                                pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 4. The instructions and the gate

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_MD = """# AGENTS.md · North Cycles

## When you start
1. Read the request and write in one sentence which behavior has to change.
2. Find the code that decides it (search) and read it before editing.

## Definition of done
- You have changed the code that decides that behavior (not just comments).
- You have added a test that checks what the request asks for, in the test file
  of the module you change (tests/test_shipping.py for shipping.py, etc.).
- You have called run_tests after your last change and they all pass.
Only then do you answer "DONE:". If a test fails, fix it and run them again.

## How to add a test
Every file in tests/ ends with `if __name__ == "__main__":`. Edit it by
replacing that line with a new class followed by that same line:

    class TaskTests(unittest.TestCase):
        def test_requested(self):
            self.assertEqual(..., ...)


    if __name__ == "__main__":

Import what you need inside the test (from shipping import shipping_cost).

## Don't touch
- data/ and docs/ unless the request asks for it.
- The tests that already exist: add yours, don't change other people's.
"""


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)


def whats_missing(log):
    """What is missing for the definition of done, according to the call log: a
    change to the code, a new test and the tests passed afterwards."""
    edits = [i for i, r in enumerate(log)
             if r["tool"] in ("edit", "create_file") and r["ok"]]
    path = lambda i: log[i]["arguments"].get("path", "")  # noqa: E731
    missing = []
    if not any(not path(i).startswith("tests/") for i in edits):
        missing.append("you haven't changed the program code")
    if not any(path(i).startswith("tests/") for i in edits):
        missing.append("you haven't added any test in tests/")
    last = edits[-1] if edits else -1
    runs = [r for r in log[last + 1:] if r["tool"] == "run_tests"]
    if not runs:
        missing.append("you haven't called run_tests after your last change")
    elif not runs[-1]["result"].get("all_pass"):
        missing.append("the last test run has failures")
    return missing

## 5. The agent

The loop of the previous book, with the optional `AGENTS.md` and the gate. The ZIP code task, first with only the request and then with the minimal workbench:

In [ ]:
def agent(tok, model, task, instructions="", gate=False, max_steps=MAX_STEPS,
          verbose=False):
    """The agent loop on a clean copy of the repository. With "instructions", the
    AGENTS.md goes after the system message; with "gate", a "DONE" without the
    definition of done is sent back with what is missing.
    Returns the row of results, already evaluated with the hidden tests."""
    start = time.time()
    system = SYSTEM + ("\n\n" + instructions if instructions else "")
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": task["request"]}]
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    with rt.Workspace() as workspace:
        tools = rt.Tools(workspace)
        for steps in range(1, max_steps + 1):
            text, trimmed = trim_conversation(tok, messages)
            if trimmed is None:
                break
            answer, n_input, n_output = generate(tok, model, text)
            n_in, n_out = n_in + n_input, n_out + n_output
            calls = read_calls(answer)
            if not calls:
                messages.append({"role": "assistant", "content": answer})
                if verbose:
                    print(f"[{steps}] {answer}")
                missing = whats_missing(tools.log) if gate else []
                if says_done(answer) and missing and rejections < MAX_REJECTIONS:
                    rejections += 1
                    notice = ("It isn't done yet according to AGENTS.md: "
                              + "; ".join(missing) + ". Keep working.")
                    messages.append({"role": "user", "content": notice})
                    if verbose:
                        print(f"    GATE: {notice}")
                    continue
                final = answer
                break
            thought = answer.split("<tool_call>")[0].strip()
            messages.append({"role": "assistant", "content": thought,
                             "tool_calls": [{"type": "function", "function": c}
                                            for c in calls if c["name"]]})
            for call in calls:
                if call["name"] is None:
                    result = {"error": "The tool call cannot be understood."}
                else:
                    result = tools.call(call["name"], call["arguments"])
                if verbose:
                    print(f"[{steps}] {call['name']}"
                          f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
                messages.append({"role": "tool", "name": call["name"] or "?",
                                 "content": json.dumps(result, ensure_ascii=False)})
        evaluation = rt.evaluate_attempt(workspace, task)
        missing = whats_missing(tools.log)
    done = final is not None and says_done(final)
    return {"task": task["id"], "type": task["meta"]["type"],
            "solved": evaluation["solved"], "says_done": done,
            "false_done": done and not evaluation["solved"],
            "meets_definition": not missing, "rejections": rejections,
            "hidden": evaluation["hidden"], "visible": evaluation["visible"],
            "out_of_scope": evaluation["out_of_scope"],
            "edited": any(r["tool"] in ("edit", "create_file") and r["ok"]
                          for r in tools.log),
            "ran_tests": any(r["tool"] == "run_tests"
                             for r in tools.log),
            "steps": steps, "input_tokens": n_in, "output_tokens": n_out,
            "seconds": round(time.time() - start, 1),
            "answer": final or "(no final answer)",
            "calls": [[r["tool"], r["arguments"].get("path", ""), r["ok"]]
                      for r in tools.log],
            "diff": evaluation["diff"]}

In [ ]:
for name, options in (("REQUEST ONLY", {}),
                      ("MINIMAL WORKBENCH", {"instructions": AGENTS_MD,
                                             "gate": True})):
    print(f"--- {name}")
    row = agent(tok, model, rt.read_task("T02"), verbose=True, **options)
    print("Solved:", row["solved"], "· Says done:", row["says_done"])
    print(row["diff"][:1500] or "(no changes)")

## 6. The fourteen tasks with both variants

By default a sample is run; with `COMPLETE = True`, the book's whole experiment.

In [ ]:
VARIANTS = {
    "request only": {},
    "minimal workbench": {"instructions": AGENTS_MD, "gate": True},
}


def summary(rows):
    """The totals of a variant."""
    return {"tasks": len(rows),
            "solved": sum(r["solved"] for r in rows),
            "says_done": sum(r["says_done"] for r in rows),
            "false_done": sum(r["false_done"] for r in rows),
            "edited": sum(r["edited"] for r in rows),
            "ran_tests": sum(r["ran_tests"] for r in rows),
            "with_out_of_scope": sum(bool(r["out_of_scope"]) for r in rows),
            "steps": sum(r["steps"] for r in rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "output_tokens": sum(r["output_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def run_all(tok, model, options, tasks=TASKS):
    """One variant with every task: its rows and its summary."""
    rows = []
    for task_id in tasks:
        row = agent(tok, model, rt.read_task(task_id), **options)
        rows.append(row)
        print(f"  {task_id} {'solved' if row['solved'] else 'no    '}"
              f"  {'says done' if row['says_done'] else 'not said '}"
              f"  steps={row['steps']:2d}  {row['seconds']:6.1f} s", flush=True)
    return {"rows": rows, "summary": summary(rows)}

In [ ]:
# With COMPLETE = False three tasks are run (about an hour on Colab's T4);
# with True, the chapter's whole experiment (about six hours on the T4).
COMPLETE = False
tasks = TASKS if COMPLETE else ["T01", "T04", "T13"]
results = {}
for name, options in VARIANTS.items():
    print(f"== {name}")
    results[name] = run_all(tok, model, options, tasks=tasks)
    print(results[name]["summary"])

The book's full results, with every task, are saved in `data/results/chapter_1/`, which `prepare` has downloaded:

In [ ]:
import json
saved = json.loads((ROOT / "data" / "results" / "chapter_1" / "results.json").read_text(encoding="utf-8"))
for name, r in saved.items():
    print(name, r["summary"])

The false "done"s of the minimal workbench: the answer, the calls and the diff. What really changed?

In [ ]:
for r in results["minimal workbench"]["rows"]:
    if r["false_done"]:
        print("✗", r["task"], r["answer"][:200])
        print("   ", [(t, path, ok) for t, path, ok in r["calls"]
                     if t in ("edit", "create_file")])
        print(r["diff"][:800])

## Exercises

The results files the exercises cite are in `data/results/chapter_1/`, which `prepare` downloads with the data.

1. For each of the twelve false "done"s of the agent with only the request, say which surface of the workbench it was missing. Is there any that does not fit any of the seven? *(No GPU.)*

2. Run the ZIP code task (T02) with `verbose=True` in both variants. At which step do the two traces split? How does the trace with the minimal workbench end, and why does the program take that answer as the end of the task? *(GPU, about 15 minutes.)*

3. Try the `AGENTS.md` without the gate, with the fourteen tasks. Are the instructions enough for the agent to run the tests? How many false "done"s are left? The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T01, T04 and T13. *(No GPU with the saved results; GPU, about 50 minutes, with those tasks.)*

4. Now try the gate without the `AGENTS.md`. The agent does not know the definition of done until the gate sends the work back. Which piece contributes more? The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T01, T04 and T13. *(No GPU with the saved results; GPU, about 15 minutes, with those tasks.)*

5. Three single-session tasks did not make it into the bank: paid bank transfers that get canceled (T06), six months of free tune-ups (T07) and promotions without consent (T08). Read their requests and run them with the minimal workbench. Are their results similar to those of the others? The results are also in `results_exercises.json`. *(GPU, about 55 minutes, or no GPU with the saved results.)*

6. Read the function the agent with only the request wrote for the low-stock task and the hidden tests of that task, which are in the folder `data/tasks/T13/hidden_tests`. Which sentences of the request does it not meet? Write a test that detects it. *(No GPU.)*

7. Change the `edit` tool so that it rejects the edits in which `old` and `new` are identical, and repeat three tasks that the minimal workbench did not solve: partial returns, Canada and Alaska, and low stock (T03, T10 and T13). Does anything change? Why? The results are also in `results_exercises.json`. *(GPU, about 45 minutes, or no GPU with the saved results.)*

8. Add to the definition of done a line that asks to reread the request sentence by sentence and to check each sentence with the test. Repeat the same three tasks. Does an instruction like this help, or is something needed that checks it? The results with the three tasks are in `results_exercises.json`; to see it for yourself, one is enough, for example T13. *(No GPU with the saved results; GPU, about 30 minutes, with that task.)*

9. With the chapter's results, compute the input tokens and the seconds per solved task with each variant. If each million input tokens cost $1 and each false "done" cost $20 of a person's work, which variant comes out cheaper? *(No GPU.)*

10. **Project.** Write an `AGENTS.md` for a repository of your own, with a definition of done that can be checked, and a gate that looks at the diff instead of the log: it must require changes outside `tests/` that are not just comments, and a new test that fails with the original code and passes with the new one. Try it with three tasks from that repository.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [SWE-agent](https://arxiv.org/abs/2405.15793) (Yang et al., 2024). [SWE-bench](https://arxiv.org/abs/2310.06770) (Jimenez et al., 2024). The [AGENTS.md](https://agents.md) convention. [Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) (Anthropic, 2025). The North Cycles store, its program and its tasks are fictional. Original code and experiments.